In [6]:
Sys.setenv(TZ = "America/Vancouver")

In [7]:
library(sf)
library(dplyr)
library(lubridate)
library(suncalc)
library(readr)
library(purrr)


In [8]:
library(sf)
library(dplyr)

divisions <- st_read("census_divison.geojson", quiet = TRUE)
divisions_valid <- st_make_valid(divisions)
sf_use_s2(FALSE)
vancouver_union <- st_union(divisions_valid)
st_write(vancouver_union, "vancouver_union.geojson", delete_dsn = TRUE)


although coordinates are longitude/latitude, st_union assumes that they are
planar



Deleting source `vancouver_union.geojson' using driver `GeoJSON'
Writing layer `vancouver_union' to data source 
  `vancouver_union.geojson' using driver `GeoJSON'
Writing 1 features with 0 fields and geometry type Polygon.


In [9]:
Sys.setenv(TZ = "America/Vancouver")

library(sf)
library(dplyr)
library(lubridate)
library(suncalc)
library(readr)
library(purrr)

vancouver_union <- st_read("vancouver_union.geojson", quiet = TRUE)

file_list <- list.files(pattern = "^crimedata_csv_AllNeighbourhoods_\\d{4}\\.csv$")

crime_data_all <- map_dfr(file_list, ~ read_csv(.x))

crime_data_clean <- crime_data_all %>%
  filter(!is.na(X), !is.na(Y)) %>%
  filter(MONTH %in% c(1, 3, 5, 7, 9, 11))

crime_sf <- st_as_sf(crime_data_clean, coords = c("X", "Y"), crs = 26910)

crime_sf <- st_transform(crime_sf, 4326)

crime_with_division <- st_join(crime_sf, divisions["name"], join = st_within)

crime_in_vancouver <- crime_with_division[
  st_within(crime_with_division, vancouver_union, sparse = FALSE),
]

crime_in_vancouver$crime_datetime <- as.POSIXct(
  paste(crime_in_vancouver$YEAR,
        crime_in_vancouver$MONTH,
        crime_in_vancouver$DAY,
        crime_in_vancouver$HOUR,
        crime_in_vancouver$MINUTE,
        sep = "-"),
  format = "%Y-%m-%d-%H-%M",
  tz = "America/Vancouver"
)

crime_in_vancouver$date_only <- as.Date(crime_in_vancouver$crime_datetime)

unique_dates <- unique(crime_in_vancouver$date_only)
sun_times <- getSunlightTimes(
  date = unique_dates,
  lat = 49.2827,
  lon = -123.1207,
  keep = c("sunrise", "sunset"),
  tz = "America/Vancouver"
) %>%
  select(date, sunrise, sunset)

crime_in_vancouver <- left_join(crime_in_vancouver, sun_times, 
                                by = c("date_only" = "date"))

crime_in_vancouver$time_category <- ifelse(
  crime_in_vancouver$crime_datetime >= crime_in_vancouver$sunrise &
    crime_in_vancouver$crime_datetime < crime_in_vancouver$sunset,
  "day",
  "night"
)

crime_in_vancouver_df <- st_set_geometry(crime_in_vancouver, NULL)
write_csv(crime_in_vancouver_df, "crime_in_vancouver_with_division.csv")

crime_day <- crime_in_vancouver %>% filter(time_category == "day")
write_csv(st_set_geometry(crime_day, NULL), "crime_day.csv")

crime_night <- crime_in_vancouver %>% filter(time_category == "night")
write_csv(st_set_geometry(crime_night, NULL), "crime_night.csv")

cat("处理完成：\n",
    "1) crime_in_vancouver_with_division.csv\n",
    "2) crime_day.csv\n",
    "3) crime_night.csv\n")


Rows: 34352 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): TYPE, HUNDRED_BLOCK, NEIGHBOURHOOD
dbl (7): YEAR, MONTH, DAY, HOUR, MINUTE, X, Y

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 34610 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): TYPE, HUNDRED_BLOCK, NEIGHBOURHOOD
dbl (7): YEAR, MONTH, DAY, HOUR, MINUTE, X, Y

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 39185 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (3): TYPE, HUNDRED_BLOCK, NEIGHBOURHOOD
dbl (7): YEAR, MONTH, DAY, HOUR, MINUTE, X, Y

ℹ Use `spec()` to retrieve the full column specification fo

处理完成：
 1) crime_in_vancouver_with_division.csv
 2) crime_day.csv
 3) crime_night.csv
